# 03. Does a late first order stop customers coming back?

## What we're testing and why

A bad review is visible, but the bigger cost may be invisible: a customer whose **first** order arrives late may never buy again.

We take every customer's first order (placed on or before 2018-02-28, so everyone has 180 days to come back) and compare the chance of a second purchase within 180 days for customers whose first order was late with those whose first order was on time. We control for order size, freight, distance, weight, promise length, state and month.

**This is a secondary test and it was expected to be underpowered.** Only about 2% of customers come back at all. The pre-registered power calculation says we can only reliably detect a drop of about 0.67 percentage points.

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyfixest as pf
from src.db import run_query

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

In [2]:
df = run_query("SELECT * FROM causal.repeat_sample")

df["price"] = df["price"].astype(float)
df["freight"] = df["freight"].astype(float)
df["weight_g"] = df["weight_g"].astype(float)
df["distance_km"] = df["distance_km"].astype(float)
df["purchase_month"] = df["purchase_month"].astype(str)

df["log_price"] = np.log(df["price"])
df["log_freight"] = np.log(1 + df["freight"])

print("Customers:", len(df))
print("Share with a late first order:", df["late_first_order"].mean())
print("Repurchase rate (180 days):", df["repurchase_180"].mean())

Customers: 55491
Share with a late first order: 0.06613685102088628
Repurchase rate (180 days): 0.019985222828927212


In [3]:
raw_means = df.groupby("late_first_order").agg(
    n_customers=("customer_unique_id", "count"),
    repurchase_rate=("repurchase_180", "mean"),
).reset_index()
raw_means

,late_first_order,n_customers,repurchase_rate
0,0,51821,0.0203
1,1,3670,0.0161


In [4]:
controls = "log_price + log_freight + distance_km + weight_g + n_items + promised_days"

fit = pf.feols("repurchase_180 ~ late_first_order + " + controls + " | customer_state + purchase_month", data=df, vcov="hetero")
fit.summary()

###

Estimation:  OLS
Dep. var.: repurchase_180, Fixed effects: customer_state + purchase_month
sample: None = all
Inference:  hetero
Observations:  55189

| Coefficient      |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:-----------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| late_first_order |     -0.003 |        0.002 |    -1.438 |      0.150 | -0.008 |   0.001 |
| log_price        |     -0.001 |        0.001 |    -1.776 |      0.076 | -0.003 |   0.000 |
| log_freight      |      0.004 |        0.002 |     1.461 |      0.144 | -0.001 |   0.008 |
| distance_km      |     -0.000 |        0.000 |    -0.922 |      0.357 | -0.000 |   0.000 |
| weight_g         |     -0.000 |        0.000 |    -1.121 |      0.262 | -0.000 |   0.000 |
| n_items          |      0.005 |        0.002 |     2.734 |      0.006 |  0.001 |   0.009 |
| promised_days    |     -0.000 |        0.000 |    -0.013 |      0.990 | -0.000 |   0.000 |
---
RMS

/Users/gandharv/Codebase/Olist/.venv/lib/python3.11/site-packages/pyfixest/estimation/formula/model_matrix.py:151: UserWarning: 2 singleton fixed effect(s) dropped from the model.
  warnings.warn(


In [5]:
t = fit.tidy()
result = pd.DataFrame([{
    "outcome": "repurchase_180",
    "estimate": t.loc["late_first_order", "Estimate"],
    "se": t.loc["late_first_order", "Std. Error"],
    "ci_low": t.loc["late_first_order", "2.5%"],
    "ci_high": t.loc["late_first_order", "97.5%"],
    "p": t.loc["late_first_order", "Pr(>|t|)"],
    "n": fit._N,
    "base_rate": df["repurchase_180"].mean(),
    "pre_registered_mde": 0.0067,
}])

result.to_csv("../outputs/tables/test3_repeat_purchase.csv", index=False)

with open("../outputs/tables/test3_repeat_purchase.md", "w") as f:
    f.write("# Test 3: Repeat purchase (secondary)\n\n")
    f.write(result.to_markdown(index=False, floatfmt=".4f"))
    f.write("\n\n# Raw repurchase rate by first-order lateness (descriptive)\n\n")
    f.write(raw_means.to_markdown(index=False, floatfmt=".4f"))

result

,outcome,estimate,se,ci_low,ci_high,p,n,base_rate,pre_registered_mde
0,repurchase_180,-0.0032,0.0022,-0.0076,0.0012,0.1505,55189,0.0200,0.0067


## What this means for the business

**Result (Test 3, secondary).** Customers whose first order was late came back within 180 days at **1.61%**, against **2.03%** for on-time first orders. After controls, the difference is **−0.32 percentage points** (95% CI −0.76 to +0.12, **p = 0.15**, N = 55,189).

**This is not statistically significant.** That was expected: the pre-registered minimum detectable effect is about 0.67 pp, and only 2% of customers ever return. The estimate points the way we'd expect, but the data is too small to separate it from zero. This is **not** evidence that lateness has no effect on loyalty. It is evidence that this dataset cannot measure it.

**For the business:** Olist's repeat rate is so low (about 2–3%) that almost all revenue comes from first-time buyers. The loyalty cost of lateness, if it exists, is small in money terms here. In the money sizing it is shown only as a range and not counted in the total.